# MCP — connect Claude to a tool server

The **Model Context Protocol (MCP)** is an open standard for exposing tools, data, and prompts to
a model through a uniform interface. Instead of hand-writing each tool, you connect to an MCP
*server* and its tools become available to Claude.

Here we run a small **local MCP server** (`server.py`, a few wildlife tools) as a subprocess over
stdio, list its tools, wrap them with the Anthropic SDK's MCP helper, and let the **tool runner**
drive the loop — so Claude uses MCP tools just like native ones.

> The MCP client path is **async** (subprocess + anyio). In this notebook we `await` the helper
> directly (Jupyter supports top-level await); in a script you'd use `asyncio.run(...)`.

**Requirements:** `ANTHROPIC_API_KEY`, and `pip install "anthropic[mcp]" mcp`.

## Setup

`_mcp_demo.py` wraps the whole flow (spawn server → list tools → tool runner) in an async `run_query`.

In [ ]:
import os
import sys

from dotenv import load_dotenv

for _p in (".", "mcp"):
    if os.path.isfile(os.path.join(_p, "_mcp_demo.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _mcp_demo import run_query

load_dotenv()

## The server's tools

`server.py` is a self-contained MCP server built with `FastMCP`. Each `@server.tool()` function
becomes an MCP tool: `list_species`, `get_species_info`, and `count_sightings`. Run it directly
and it speaks MCP over stdio; here our client launches it as a subprocess.

## Ask a question — Claude uses the MCP tools

`run_query` spawns the server, exposes its tools, and runs the agentic loop. We `await` it. Watch
the transcript: Claude calls the MCP tools and composes an answer from their results.

In [ ]:
result = await run_query(
    "What can you tell me about the river otter, and how many sightings does it have?")

print("MCP tools available:", result["tools"])
for e in result["transcript"]:
    if e["kind"] == "text":
        if e["text"].strip():
            print("assistant:", e["text"][:200])
    else:
        print(f"  → MCP tool {e['name']}({e['input']})")

In [ ]:
from IPython.display import Markdown

Markdown(result["final"])

## Local vs remote MCP

This example uses a **local stdio server** (a subprocess you control) — great for self-contained
tools and full control of the connection. The Claude API also supports **remote MCP servers**
directly via the `mcp_servers` request parameter:

```python
response = client.beta.messages.create(
    model="claude-sonnet-4-5", max_tokens=1024,
    messages=[{"role": "user", "content": "..."}],
    mcp_servers=[{"type": "url", "url": "https://your-mcp-server.example/mcp", "name": "tools"}],
    betas=["mcp-client-2025-04-04"],
)
```

Use the SDK's MCP helpers (as here) for **local** servers, prompts, and resources, or more control
over the connection; use `mcp_servers` to let Claude connect to a **hosted** server (needs a URL,
and possibly auth).

## Notes

- **Tools come from the server.** `session.list_tools()` returns the server's tool schemas;
  `async_mcp_tool(tool, session)` adapts each one for the tool runner. Adding a tool to the server
  exposes it to Claude with no client change.
- **Async + subprocess.** The MCP client is async and launches the server process — hence
  `await` / `asyncio.run`. (That's also why this topic has no Streamlit app.)
- **Beyond tools.** MCP also carries **prompts** and **resources**; the SDK has helpers
  (`mcp_message`, `mcp_resource_to_content`) to bring those into a request.
- **Same loop.** Once adapted, MCP tools run through the exact tool-use loop from the
  `tool_use/` topic — MCP is about *where tools come from*, not a different execution model.